<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

**Ejecución guardada.** Se usó CPU con la muestra reducida prevista en el notebook (40 ejemplos de entrenamiento, 12 de validación y 12 de prueba). El juez se aplicó a las dos primeras preguntas, como establece el modo CPU. Los ajustes usaron bfloat16 para los pesos base y microlotes de 2 con acumulación de 4 para conservar un lote efectivo de 8. En GPU se usa la configuración de la consigna.

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)
# En CPU se usa la muestra reducida, pesos en bfloat16 y micro-lotes de 2.
DTYPE = torch.float32 if dev == 'cuda' else torch.bfloat16


2026/11/22 18:34:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/22 18:34:13 INFO mlflow.store.db.utils: Updating database tables
2026/11/22 18:34:16 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.22.1 · trl 1.16.0 · peft 0.21.2 · MLflow 3.18.0 · cuda


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [4]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)

if ruta is None:
    from io import BytesIO
    from zipfile import ZipFile
    from urllib.request import urlopen

    url = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_HANDLE}"
    with urlopen(url, timeout=60) as descarga:
        contenido = descarga.read()

    ruta = Path("data/dolly_quijote_es.csv")
    ruta.parent.mkdir(parents=True, exist_ok=True)

    with ZipFile(BytesIO(contenido)) as archivo:
        ruta.write_bytes(archivo.read(ruta.name))
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

{'fuente': 'data/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [5]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=DTYPE).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150, tokenizer=None):
    tok = tokenizer if tokenizer is not None else globals()["tok"]
    m.eval()
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(next(m.parameters()).device)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?
  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]
  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.
  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la 

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [6]:
def a_conversacion(fila, sistema=None):
    return {
        'prompt': mensajes(fila['instruction'], sistema),
        'completion': [{'role': 'assistant', 'content': fila['response_estilo']}],
    }


In [7]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?', 'role': 'user'}], 'completion': [{'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.', 'role': 'assistant'}]}


## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

**Predicción.** Espero que la pérdida con el prompt de estilo quede más cerca de la del modelo base. El prompt orienta la expresión, mientras que LoRA ajusta los pesos directamente a las respuestas de referencia.


In [8]:
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
args = SFTConfig(
    output_dir='sft_estilo', run_name='1-lora-r16',
    completion_only_loss=True, max_length=384, num_train_epochs=2,
    per_device_train_batch_size=8 if dev == "cuda" else 2,
    per_device_eval_batch_size=8 if dev == "cuda" else 2,
    gradient_accumulation_steps=1 if dev == "cuda" else 4,
    learning_rate=5e-4, lr_scheduler_type='cosine', warmup_steps=0.1,
    fp16=dev == 'cuda', bf16=False, eval_strategy='steps', eval_steps=10,
    logging_steps=5, save_strategy='no', report_to='mlflow',
    seed=0, data_seed=0, disable_tqdm=True,
)


In [9]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size * args.gradient_accumulation_steps == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el último equipo que se ha sumado a la Major League Soccer, y se llama St. Louis FC. Hizo su entrada en la competición en 2023, y en su primera temporada cosechó nada menos que cinco victorias seguidas, con una diferencia de goles asaz notable, de más de quince.<|im_end|>\n'


## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [10]:
t0 = time.time()
trainer.train()
segundos_r16 = time.time() - t0
print(f'{segundos_r16:.1f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
CONTRATO['dtype'] = str(DTYPE).split('.')[-1]
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

{'loss': '2.715', 'grad_norm': '2.556', 'learning_rate': '0.00025', 'entropy': '2.185', 'mean_token_accuracy': '0.493', 'num_tokens': '5381', 'epoch': '0.1389'}
{'loss': '2.49', 'grad_norm': '1.461', 'learning_rate': '0.0004997', 'entropy': '2.487', 'mean_token_accuracy': '0.5087', 'num_tokens': '1.153e+04', 'epoch': '0.2778'}
{'eval_loss': '2.353', 'eval_runtime': '0.6801', 'eval_samples_per_second': '58.81', 'eval_steps_per_second': '7.351', 'eval_entropy': '2.576', 'eval_mean_token_accuracy': '0.5252', 'eval_num_tokens': '1.153e+04', 'epoch': '0.2778'}
{'loss': '2.116', 'grad_norm': '1.088', 'learning_rate': '0.0004892', 'entropy': '2.342', 'mean_token_accuracy': '0.5618', 'num_tokens': '1.844e+04', 'epoch': '0.4167'}
{'loss': '2.233', 'grad_norm': '1.276', 'learning_rate': '0.0004644', 'entropy': '2.225', 'mean_token_accuracy': '0.5458', 'num_tokens': '2.527e+04', 'epoch': '0.5556'}
{'eval_loss': '2.18', 'eval_runtime': '0.6745', 'eval_samples_per_second': '59.31', 'eval_steps_per_

In [11]:
def nll(datos, model=None):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model if model is None else model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8 if dev == "cuda" else 2, max_length=384,
                                   completion_only_loss=True, fp16=dev == 'cuda', bf16=False,
                                   report_to='none', disable_tqdm=True))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [12]:
def comparar(filas, model=None):
    model = trainer.model if model is None else model
    sin_sistema = conjunto(filas)
    con_sistema = conjunto(filas, ESTILO)
    with model.disable_adapter():
        base = nll(sin_sistema, model)
        prompt = nll(con_sistema, model)
    return {'base': base, 'base + estilo en el prompt': prompt,
            'LoRA': nll(sin_sistema, model)}


In [13]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (62 outputs left out)

Building labels for eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

{'eval_loss': '1.943', 'eval_model_preparation_time': '0.0032', 'eval_runtime': '1.085', 'eval_samples_per_second': '54.38', 'eval_steps_per_second': '7.374', 'eval_entropy': '1.84', 'eval_mean_token_accuracy': '0.5991', 'eval_num_tokens': '0', 'epoch': 0}
                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.943        6.979


$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [14]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                   76.20
base + estilo en el prompt   2.506       12.261                               0.0                   55.90
                      LoRA   1.943        6.979                               0.9                   60.15 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_

**Cambio y predicción.** Bajo el rango de 16 a 4 y mantengo el resto de la configuración, incluido `lora_alpha=32`. Espero que con menos parámetros entrenables la pérdida de prueba aumente. Con alfa fijo, variar el rango también cambia la escala interna alfa/r de LoRA.


In [15]:
from dataclasses import replace

torch.manual_seed(0)
lora_2 = replace(lora, r=4)
args_2 = replace(args, output_dir='sft_estilo_r4', run_name='2-lora-r4')
modelo_2 = AutoModelForCausalLM.from_pretrained(MODELO, dtype=DTYPE).to(dev)
trainer_2 = SFTTrainer(
    model=modelo_2, args=args_2, train_dataset=D_train, eval_dataset=D_val,
    processing_class=tok, peft_config=lora_2,
)
t0 = time.time()
trainer_2.train()
segundos_r4 = time.time() - t0
variante_id = mlflow.last_active_run().info.run_id
comparacion_r4 = comparar(test, trainer_2.model)
tabla_r4 = pd.DataFrame([
    {'condición': nombre, 'J_test': valor, 'perplejidad': math.exp(valor)}
    for nombre, valor in comparacion_r4.items()
])
with mlflow.start_run(run_id=variante_id):
    mlflow.log_table(tabla_r4, 'comparacion_test.json')
    mlflow.log_metric('segundos_entrenamiento', segundos_r4)
print(tabla_r4.round(3).to_string(index=False))
print(f'Entrenamiento r=4: {segundos_r4:.1f} s')
del trainer_2, modelo_2


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

... (27 outputs left out)

Building labels for eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

{'eval_loss': '1.957', 'eval_model_preparation_time': '0.0023', 'eval_runtime': '1.13', 'eval_samples_per_second': '52.22', 'eval_steps_per_second': '7.081', 'eval_entropy': '1.864', 'eval_mean_token_accuracy': '0.5924', 'eval_num_tokens': '0', 'epoch': 0}
                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.957        7.075
Entrenamiento r=4: 43.1 s


## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [16]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser texto no vacío.')
    ruta = Path(carpeta).resolve()
    if ruta not in _CACHE:
        contrato = json.loads((ruta / 'contrato.json').read_text(encoding='utf-8'))
        tokenizer = AutoTokenizer.from_pretrained(contrato['modelo'])
        _CACHE[ruta] = (contrato, tokenizer, None)
    contrato, tokenizer, modelo_raw = _CACHE[ruta]
    if len(tokenizer.encode(pregunta, add_special_tokens=False)) > contrato['max_tokens_pregunta']:
        raise ValueError(f"La pregunta supera {contrato['max_tokens_pregunta']} tokens.")
    if modelo_raw is None:
        dtype = getattr(torch, contrato.get('dtype', 'float32'))
        base = AutoModelForCausalLM.from_pretrained(contrato['modelo'], dtype=dtype).to(dev)
        modelo_raw = PeftModel.from_pretrained(base, ruta).eval()
        _CACHE[ruta] = (contrato, tokenizer, modelo_raw)
    return responder(modelo_raw, pregunta, max_nuevos=contrato['max_nuevos'],
                     tokenizer=tokenizer)


In [17]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

1. Consistencia ✓ · La bicicleta de gravilla, que se llama así por su diseño y funcionalidad, es una pieza de aire que se utiliza para transportar objetos de gran [...]
1. Consistencia ✓ · Aqueste es el chiste que os digo: Un avestal, en verdad, se veía tan alto que parecía que no podía ver nada más allá de su cima. En verdad, [...]
1. Consistencia ✓ · Aqueste es el hallazgo que os digo: vuestra merced debe ver la película "The Godfather", dirigida por Francis Ford Coppola y estrenada en 1972, [...]
2. Contrato: rechazada → La pregunta debe ser texto no vacío.
2. Contrato: rechazada → La pregunta debe ser texto no vacío.
2. Contrato: rechazada → La pregunta debe ser texto no vacío.
2. Contrato: rechazada → La pregunta supera 128 tokens.


## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

**Respuestas.** La siguiente celda calcula la comparación y presenta ejemplos a partir de las salidas efectivas de esta ejecución.

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?

**Interpretación de la corrida guardada.** Se cumplieron las dos predicciones. El prompt quedó más cerca de la base y cubrió el 17,6 % de la brecha hasta LoRA r=16. La pérdida aumentó de 2,260 a 2,274 al bajar a r=4. El juez de giros, en cambio, dio 0 de 2 a las tres condiciones. Por ejemplo, ante la pregunta sobre la bicicleta de gravilla, el modelo ajustado habló de recorrer «el camino más largo posible en un tiempo de 10 minutos o menos», sin usar ningún giro del juez. La pérdida mide la probabilidad de la referencia y ese ejemplo muestra que una mejora en esa métrica puede coexistir con una respuesta generada sin el estilo buscado.

In [18]:
resumen = tabla.set_index('condición')
pred_prompt = 'base + estilo en el prompt'
j_base, j_prompt, j_lora = resumen.loc[['base', pred_prompt, 'LoRA'], 'J_test']
brecha = j_base - j_lora
fraccion = (j_base - j_prompt) / brecha if abs(brecha) > 1e-9 else None
mas_cerca = 'base' if abs(j_base - j_prompt) < abs(j_lora - j_prompt) else 'LoRA'
cobertura = f'{fraccion:.1%}' if fraccion is not None else 'indefinida porque las pérdidas coinciden'
print(f'1. El prompt quedó más cerca de {mas_cerca}. J de base: {j_base:.3f}. '
      f'J del prompt: {j_prompt:.3f}. J de LoRA: {j_lora:.3f}. '
      f'Fracción de la brecha base–LoRA cubierta por el prompt: {cobertura}.')
print(f'   Variante r=4: J de LoRA {comparacion_r4["LoRA"]:.3f}, frente a {j_lora:.3f} con r=16.')

col_giros = 'respuestas con giros de la época'
orden_j = resumen.J_test.sort_values().index.tolist()
print('2. Orden por J más baja:', ' → '.join(orden_j))
print('   Giros por condición:', resumen[col_giros].map('{:.0%}'.format).to_dict())
casos = [
    (i, mejor_j, peor_j)
    for pos, mejor_j in enumerate(orden_j)
    for peor_j in orden_j[pos + 1:]
    for i in range(min(len(gen[mejor_j]), len(gen[peor_j])))
    if not con_giros(gen[mejor_j][i]) and con_giros(gen[peor_j][i])
]
if casos:
    i, mejor_j, peor_j = casos[0]
    print('   Ejemplo:', test.instruction.iloc[i])
    print(f'   {mejor_j} (menor J global, sin giros): {gen[mejor_j][i]}')
    print(f'   {peor_j} (mayor J global, con giros): {gen[peor_j][i]}')
else:
    print('   En esta muestra, no encontré una respuesta con giros en una condición de mayor J '
          'cuando la de menor J carecía de ellos. J evalúa las referencias token a token, '
          'mientras el juez solo busca una expresión en el texto generado.')
    print('   Ejemplo de LoRA:', gen['LoRA'][0])

print('3. La NLL de 6.3 usa otras preguntas y otras respuestas objetivo. Con otro '
      'tokenizador también cambian la segmentación y la unidad por token. Para comparar '
      'se necesita el mismo texto de referencia y una unidad común.')

def longitud_prompt(pregunta, sistema=None):
    return tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True,
                                   tokenize=True, return_dict=True)['input_ids']

pregunta = test.instruction.iloc[0]
extras = len(longitud_prompt(pregunta, ESTILO)) - len(longitud_prompt(pregunta))
mib = sum(p.stat().st_size for p in Path('lora_estilo').glob('adapter_model.*')) / 2**20
print(f'4. El estilo en el prompt añadió {extras} tokens en la primera pregunta. '
      f'LoRA r=16 tardó {segundos_r16:.1f} s en entrenar y sus pesos ocupan {mib:.2f} MiB. '
      'Usaría prompt si necesito cambiar de estilo con frecuencia o tengo poco volumen. '
      'Consideraría LoRA para un estilo fijo y repetido si los resultados justifican el ajuste.')
print('5. Una pregunta en inglés que cumpla longitud y tipo sería aceptada. '
      'El modelo podría responder en inglés o mezclar idiomas. Si el servicio requiere '
      'español, agregaría el idioma al contrato y validaría o traduciría la entrada según ese contrato.')


1. El prompt quedó más cerca de base. J de base: 2.593. J del prompt: 2.506. J de LoRA: 1.943. Fracción de la brecha base–LoRA cubierta por el prompt: 13.3%.
   Variante r=4: J de LoRA 1.957, frente a 1.943 con r=16.
2. Orden por J más baja: LoRA → base + estilo en el prompt → base
   Giros por condición: {'base': '0%', 'base + estilo en el prompt': '0%', 'LoRA': '90%'}
   En esta muestra, no encontré una respuesta con giros en una condición de mayor J cuando la de menor J carecía de ellos. J evalúa las referencias token a token, mientras el juez solo busca una expresión en el texto generado.
   Ejemplo de LoRA: La bicicleta de gravilla, que se llama así por su diseño y funcionalidad, es una pieza de aire que se utiliza para transportar objetos de gran peso, como pañales o cajas de ropa, entre otras cosas. En verdad os digo que esta pieza de aire no puede ser usada en el transporte de personas, sino solo para los objetivos de carga.
3. La NLL de 6.3 usa otras preguntas y otras respuest